This example fits the degree-9 polynomial model to the noisy sinusoid by **stochastic gradient descent (SGD)** on the *regularized* empirical risk, the same ridge objective solved in closed form on the [linear regression](/aiml-common/lectures/regression/linear-regression) page. There the regularization strength was tuned by sweeping the closed-form solution and reading off the value $\lambda^\ast \approx 3.04\times10^{-3}$ that minimizes held-out error. Here you reach the same kind of solution iteratively, and you reuse that $\lambda^\ast$ to anchor the choice of regularization rather than re-tuning from scratch.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()

## The dataset

You use the identical training set as the closed-form page: ten points of $\sin(2\pi x)$ on $[0,1]$ corrupted by Gaussian noise with standard deviation $0.25$. Holding the data fixed is what lets the value $\lambda^\ast$ carry over unchanged.

In [ ]:
def sinusoidal(x):
    return np.sin(2 * np.pi * x)

def create_toy_data(func, sample_size, std, domain=[0, 1]):
    x = np.linspace(domain[0], domain[1], sample_size)
    np.random.shuffle(x)
    y = func(x) + np.random.normal(scale=std, size=x.shape)
    return x, y

np.random.seed(1)                       # same draw as the closed-form page
x_train, y_train = create_toy_data(sinusoidal, 10, 0.25)

In [ ]:
plt.figure(figsize=[10, 8])
plt.scatter(x_train, y_train, facecolor="none", edgecolor="b", s=50, label="training data")
xg = np.linspace(0, 1, 200)
plt.plot(xg, sinusoidal(xg), "-g", label="target function")
plt.xlabel("$x$"); plt.ylabel("$y$"); plt.legend()
plt.show()

## Standardized polynomial features

The model is a degree-9 polynomial, $g(x;\boldsymbol{\theta}) = \bar{y} + \sum_{k=1}^{9}\theta_k\,\phi_k(x)$. Raw monomials $x, x^2, \dots, x^9$ on $[0,1]$ span many orders of magnitude, which makes the gradient steps lopsided and the penalty $\lambda\lVert\boldsymbol{\theta}\rVert^2$ act unevenly across coordinates. Standardizing each feature to zero mean and unit variance puts every coordinate on the same footing, so a single learning rate and a single $\lambda$ are meaningful and the value of $\lambda^\ast$ transfers directly from the closed-form page, which uses the same standardized features. The intercept is absorbed by centering the targets at $\bar{y}$.

In [ ]:
M = 9
P_train = np.vander(x_train, M + 1, increasing=True)[:, 1:]   # columns x^1 .. x^9
mu, sd = P_train.mean(0), P_train.std(0) + 1e-12

def featurize(xq):
    P = np.vander(np.ravel(xq), M + 1, increasing=True)[:, 1:]
    return (P - mu) / sd

Phi_train = featurize(x_train)
y_bar = y_train.mean()                  # intercept; SGD fits the centered residual

# held-out validation set for scoring lambda (same construction as the closed-form page)
x_val = np.linspace(0, 1, 100)
y_val = sinusoidal(x_val) + np.random.RandomState(0).normal(scale=0.25, size=x_val.size)
Phi_val = featurize(x_val)

## The regularized objective and the SGD update

SGD minimizes the same ridge objective the closed-form page solves exactly, the sum of squared residuals plus an $\ell_2$ penalty,

$$J(\boldsymbol{\theta}) = \sum_{i=1}^{n}\big(\boldsymbol{\phi}_i^{\top}\boldsymbol{\theta} - \tilde{y}_i\big)^2 + \lambda\lVert\boldsymbol{\theta}\rVert^2, \qquad \tilde{y}_i = y_i - \bar{y}.$$

Its minimizer satisfies the normal equations $(\boldsymbol{\Phi}^{\top}\boldsymbol{\Phi} + \lambda\mathbf{I})\boldsymbol{\theta} = \boldsymbol{\Phi}^{\top}\tilde{\mathbf{y}}$, so using this convention makes $\lambda$ here identical to the closed-form $\lambda^\ast$. Each SGD step draws a mini-batch $\mathcal{B}$ of size $B$ and follows an unbiased estimate of the full gradient, scaling the batch sum by $n/B$:

$$\boldsymbol{\theta} \leftarrow \boldsymbol{\theta} - \eta\Big(\tfrac{2n}{B}\,\boldsymbol{\Phi}_{\mathcal{B}}^{\top}(\boldsymbol{\Phi}_{\mathcal{B}}\boldsymbol{\theta} - \tilde{\mathbf{y}}_{\mathcal{B}}) + 2\lambda\boldsymbol{\theta}\Big).$$

In [ ]:
def sgd_fit(lam, lr=0.004, epochs=20000, batch_size=5, seed=0):
    g = np.random.default_rng(seed)
    yc = y_train - y_bar                # centered target
    n = len(y_train)
    theta = np.zeros(M)
    train_hist, val_hist = [], []
    for _ in range(epochs):
        order = g.permutation(n)
        for s in range(0, n, batch_size):
            b = order[s:s + batch_size]
            err = Phi_train[b] @ theta - yc[b]
            grad = (2 * n / len(b)) * (Phi_train[b].T @ err) + 2 * lam * theta
            theta -= lr * grad
        res = Phi_train @ theta - yc
        train_hist.append(np.sum(res**2) + lam * np.sum(theta**2))
        val_hist.append(np.mean((Phi_val @ theta + y_bar - y_val) ** 2))
    return theta, np.array(train_hist), np.array(val_hist)

## Choosing $\lambda$: a chicken-and-egg shortcut

Running SGD needs a value of $\lambda$, yet $\lambda$ is itself a hyperparameter you are supposed to choose by comparing held-out error across candidates. That circularity is the chicken-and-egg: you cannot run a fit until you commit to a $\lambda$, but you cannot score a $\lambda$ until you run the fit. A full search over many decades of $\lambda$, each one a complete SGD run, is expensive.

The closed-form page already broke the circle once, locating $\lambda^\ast \approx 3.04\times10^{-3}$ on this exact data. You reuse that result: first fit SGD at $\lambda^\ast$ itself, then search only a narrow band around it. Restricting the search to one decade either side of $\lambda^\ast$ is the shortcut, you trust the closed-form page to have found the right neighborhood.

In [ ]:
LAMBDA_STAR = 3.04e-3        # optimum of the closed-form ridge fit (companion page)

theta_sgd, train_hist, val_hist = sgd_fit(LAMBDA_STAR)

# closed-form ridge at the same lambda, for comparison
theta_cf = np.linalg.solve(
    Phi_train.T @ Phi_train + LAMBDA_STAR * np.eye(M), Phi_train.T @ (y_train - y_bar))

xq = np.linspace(0, 1, 200)
gap = np.max(np.abs(featurize(xq) @ theta_sgd - featurize(xq) @ theta_cf))
print(f"validation MSE at lambda*           : {val_hist[-1]:.4f}")
print(f"max |SGD - closed-form| over [0, 1] : {gap:.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=[14, 6])
ax[0].plot(train_hist, color="b", label="training objective $J(\\theta)$")
ax[0].plot(val_hist, color="r", label="validation MSE")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend()
ax[0].set_title(r"SGD at $\lambda^\ast$")

xg = np.linspace(0, 1, 200); Phi_g = featurize(xg)
ax[1].scatter(x_train, y_train, facecolor="none", edgecolor="b", s=50, label="training data")
ax[1].plot(xg, sinusoidal(xg), "-g", label="target function")
ax[1].plot(xg, Phi_g @ theta_sgd + y_bar, "-m", lw=2, label=r"SGD fit ($\lambda^\ast$)")
ax[1].plot(xg, Phi_g @ theta_cf + y_bar, "--k", lw=1.5, label="closed-form ridge")
ax[1].set_xlabel("$x$"); ax[1].set_ylabel("$y$"); ax[1].legend()
ax[1].set_title("Regularized degree-9 fit")
plt.tight_layout(); plt.show()

## Searching a narrow band around $\lambda^\ast$

Now treat $\lambda$ as the quantity to optimize, but only over a narrow log-range bracketing $\lambda^\ast$, one decade either side. Each trial runs a full SGD fit and reports the best validation MSE, and the search keeps the $\lambda$ that minimizes it.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    lam = trial.suggest_float("lambda", LAMBDA_STAR / 10, LAMBDA_STAR * 10, log=True)
    _, _, val_hist = sgd_fit(lam)
    return val_hist.min()

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

print(f"lambda* (closed form) : {LAMBDA_STAR:.2e}")
print(f"best lambda (SGD)     : {study.best_params['lambda']:.2e}")
print(f"best validation MSE   : {study.best_value:.4f}")

In [ ]:
lams = np.array([t.params["lambda"] for t in study.trials])
vals = np.array([t.value for t in study.trials])
order = np.argsort(lams)
plt.figure(figsize=[10, 8])
plt.plot(lams[order], vals[order], "-o", label="validation MSE")
plt.axvline(LAMBDA_STAR, color="g", ls="--", label=r"$\lambda^\ast$ (closed form)")
plt.axvline(study.best_params["lambda"], color="m", ls=":", label="best $\lambda$ (SGD)")
plt.xscale("log"); plt.xlabel(r"$\lambda$"); plt.ylabel("validation MSE"); plt.legend()
plt.show()

## Takeaways

- SGD minimizes the **regularized** empirical risk: the penalty enters the gradient as $2\lambda\boldsymbol{\theta}$, shrinking the weights every step and curbing the degree-9 overfitting. At $\lambda^\ast$ the SGD curve sits almost on top of the closed-form ridge fit.
- Standardizing the features and adopting the sum-of-squares convention make a single $\lambda$ meaningful and let $\lambda^\ast$ transfer unchanged from the closed-form page.
- The narrow search lands slightly *below* $\lambda^\ast$. This is expected: $\lambda^\ast$ was tuned for the fully converged least-squares solution, whereas iterative SGD adds its own implicit regularization, so it needs a little less explicit shrinkage. Anchoring the search to $\lambda^\ast$ still puts you in the right neighborhood, which is the whole point of the shortcut.

**Key references**: [@Keskar2016-io; @Bottou2016-yl; @Andrychowicz2016-lv]